# EDA on Academic Papers

## Datasets

### s2orc
"This dataset contains titles, abstracts, and citations from scientific papers from the Semantic Scholar Open Research Corpus (S2ORC). This dataset can and has been used to train embedding models, and works out of the box to train or finetune Sentence Transformer models."
- [sentence-transformers/s2orc](https://huggingface.co/datasets/sentence-transformers/s2orc)

### peS2o
"The peS2o dataset is a collection of ~40M creative open-access academic papers, cleaned, filtered, and formatted for pre-training of language models. <br>
... <br>
peS2o V2 (Latest) <br>
Key Facts <br>
Knowledge cutoff: 2023-01-03 <br>
Number of documents: 38.97M <br>
Number of whitespace-separated tokens*: 42.01B"

\- [allenai/peS2o](https://huggingface.co/datasets/allenai/peS2o)

> **Note**: This is a cleaned version of s2orc. However, s2orc contains more metadata (`title`, `abstract`, `citation`). Also `id` in `peS2o` corresponds to the order in `s2orc`.

## Setup

### Imports

In [6]:
import pandas as pd
import random as rand
from datasets import load_dataset
from transformers import AutoTokenizer

### Load Data

In [10]:
base_s2 = "hf://datasets/sentence-transformers/s2orc/title-citation-pair-all"
base_pe = "hf://datasets/allenai/peS2o/data/v2"
data_files_s2 = f"{base_s2}/*.arrow"
data_files_pe = {
    "train": f"{base}/train-0000*-of-00020.json.gz",
    "validation": f"{base}/validation-0000*-of-00002.json.gz",
}
s2_ds = load_dataset("arrow", data_files = data_files_s2, streaming=True)
pe_ds = load_dataset("json", data_files=data_files, streaming=True)

Resolving data files:   0%|          | 0/334 [00:00<?, ?it/s]

In [13]:
s2_ds

IterableDatasetDict({
    train: IterableDataset({
        features: ['title', 'citation'],
        num_shards: 334
    })
})

In [14]:
pe_ds

IterableDatasetDict({
    train: IterableDataset({
        features: ['id', 'source', 'version', 'added', 'created', 'text'],
        num_shards: 10
    })
    validation: IterableDataset({
        features: ['id', 'source', 'version', 'added', 'created', 'text'],
        num_shards: 2
    })
})

In [17]:
sample_s2 = list(s2_ds["train"].take(1000))
df_s2 = pd.DataFrame(sample_s2)

In [18]:
sample_pe = list(pe_ds["train"].take(1000))
df_pe = pd.DataFrame(sample_pe)

In [21]:
print(df_s2.columns)
print(df_s2.head())

Index(['title', 'citation'], dtype='str')
                                               title  \
0  SugarBindDB, a resource of glycan-mediated hos...   
1  SugarBindDB, a resource of glycan-mediated hos...   
2  SugarBindDB, a resource of glycan-mediated hos...   
3  SugarBindDB, a resource of glycan-mediated hos...   
4  SugarBindDB, a resource of glycan-mediated hos...   

                                            citation  
0  WURCS: The Web3 Unique Representation of Carbo...  
1  The Genomes OnLine Database (GOLD) v.5: a meta...  
2  Three-dimensional representations of complex c...  
3   Introducing glycomics data into the Semantic Web  
4  SugarBindDB SugarBindDB : Resource of Pathogen...  


In [22]:
print(df_pe.columns)
print(df_pe.head())
print()
print(df_pe.drop(columns=["text"]))

Index(['id', 'source', 'version', 'added', 'created', 'text'], dtype='str')
          id      source version                     added  \
0   40474210  s2ag/train      v2  2018-04-03T04:47:30.209Z   
1  145156110  s2ag/train      v2  2019-05-06T14:06:07.318Z   
2   24962770  s2ag/train      v2  2018-04-03T04:06:09.115Z   
3   99777970  s2ag/train      v2  2019-04-08T13:08:02.582Z   
4  212678020  s2ag/train      v2  2020-03-13T13:04:57.328Z   

                    created                                               text  
0  2011-08-01T00:00:00.000Z  [Short-term effectiveness of anterior cruciate...  
1  1999-10-01T00:00:00.000Z  Appraisal of the Homogeneity of Error Variance...  
2  1996-06-01T00:00:00.000Z  Two-phase helical CT for pancreatic tumors: pa...  
3  2001-02-05T00:00:00.000Z  Stability issues of conjugated polymer/fullere...  
4  2020-03-12T00:00:00.000Z  Influence of sodium hypochlorite/etidronic aci...  

            id      source version                     added  \


## Text Statistics

### Word Count

In [14]:
df["w_count"] = df["text"].str.split(" ").str.len()

In [15]:
df["w_count"].describe()

count    1000.000000
mean      193.813000
std       101.289731
min        54.000000
25%       126.000000
50%       172.500000
75%       239.000000
max       864.000000
Name: w_count, dtype: float64

### Token Count

In [6]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
encoded_input = tokenizer(df["text"].to_list(), add_special_tokens=False)
df["t_count"] = [len(ids) for ids in encoded_input["input_ids"]]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1081 > 1024). Running this sequence through the model will result in indexing errors


In [7]:
df["t_count"].describe()

count    1000.000000
mean      277.803000
std       153.747366
min        76.000000
25%       174.000000
50%       241.500000
75%       344.250000
max      1368.000000
Name: t_count, dtype: float64

### Estimated Total Tokens

In [8]:
df["t_count"].median() * len(ds)

np.float64(483.0)

## Examples 

In [41]:
def show_examples(n=2):
    for _ in range(n):
        idx = rand.randint(0,len(df))
        text = df["text"][idx]
        num_words = df["w_count"][idx]
        num_tokens = df["t_count"][idx]
        print(f"Index: {idx}")
        print(text)
        print(f"Word Count:  {num_words} \nToken Count: {num_tokens}")
        print("\n")

In [43]:
show_examples(3)

Index: 566
Autocatalytic Pyrolysis of Wastewater Biosolids for Product Upgrading.

The main goals for sustainable water resource recovery include maximizing energy generation, minimizing adverse environmental impacts, and recovering beneficial resources. Wastewater biosolids pyrolysis is a promising technology that could help facilities reach these goals because it produces biochar that is a valuable soil amendment as well as bio-oil and pyrolysis gas (py-gas) that can be used for energy. The raw bio-oil, however, is corrosive; therefore, employing it as fuel is challenging using standard equipment. A novel pyrolysis process using wastewater biosolids-derived biochar (WB-biochar) as a catalyst was investigated to decrease bio-oil and increase py-gas yield for easier energy recovery. WB-biochar catalyst increased the py-gas yield nearly 2-fold, while decreasing bio-oil production. The catalyzed bio-oil also contained fewer constituents based on GC-MS and GC-FID analyses. The energy shif

## String Parsing

In [55]:
print(f'Contains "Abstract":     {df["text"].str.contains("Abstract").sum()}')
print(f'Contains "Conclusion":   {df["text"].str.contains("Conclusion").sum()}')
print(f"Total:                   {len(df)}")

Contains "Abstract":     52
Contains "Conclusion":   40
Total:                   1000


In [59]:
print("New Line Counts:")
df["text"].str.count("\n").describe()

New Line Counts:


count    1000.000000
mean        2.936000
std         3.673608
min         2.000000
25%         2.000000
50%         2.000000
75%         2.000000
max        31.000000
Name: text, dtype: float64

## **TODO**: combine with other dataset to get metadata (title, etc.)
*Idea*: Try converting to pyspark RDDs and do a left inner join

*Note*: If titles/citations are helpful